## Model Initialize

In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatGoogleGenerativeAI(model = "gemini-3.5-flash-lite",
                                max_tokens = 512,
                                api_key = os.getenv("GOOGLE_API_KEY"))

In [35]:
import pandas as pd

data = {
    "Product": [
        "Wireless Headphones",
        "Coffee Maker",
        "Running Shoes",
        "Smartphone Case",
        "Laptop Stand"
    ],
    "Review": [
        "The sound quality is excellent, but the battery could last longer.",
        "Easy to use and makes great coffee. The water tank is a little small.",
        "Very comfortable for long walks, but they run slightly small.",
        "Good protection and nice design, but the buttons are hard to press.",
        "Strong and stable. It improved my desk setup a lot."
    ]
}

df = pd.DataFrame(data)

df.head()

,Product,Review
0,Wireless Headphones,"The sound quality is excellent, but the batter..."
1,Coffee Maker,Easy to use and makes great coffee. The water ...
2,Running Shoes,"Very comfortable for long walks, but they run ..."
3,Smartphone Case,"Good protection and nice design, but the butto..."
4,Laptop Stand,Strong and stable. It improved my desk setup a...


## Chains

In [36]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

### 1) Simple Sequential Chain (One Input -> One Output)

In [37]:
prompt1 = ChatPromptTemplate.from_template(
    """
    Give me ONE creative company name for a company that makes {product}.
    Return ONLY the company name.
    Do not explain anything.
    """
)

chain1 = prompt1 | model | StrOutputParser()

In [38]:
response = chain1.invoke({"product":df['Product'][0]})

In [39]:
response

'AuraFree'

In [40]:
prompt2 = ChatPromptTemplate.from_template(
    """
    Write ONE short company description for a company named "{company_name}".
    Requirements:
        - Exactly ONE sentence.
        - Maximum 20 words.
        - Do not provide options.
        - Do not explain your answer.
        - Return ONLY the description.
    """
)

chain2 = prompt2 | model | StrOutputParser()

In [41]:
chain = (
    {"company_name": chain1,}
    | prompt2
    | model
    | StrOutputParser()
)

In [42]:
chain.invoke({
    "product": df['Product'][0]
})

'AetherAudio crafts immersive, wireless soundscapes that elevate your everyday listening experience into pure auditory art.'

### 2) Complex Sequential Chain (Multiple Input -> Multiple Output)

In [54]:
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel
)
from operator import itemgetter

In [44]:
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)

translation_chain = (
    translate_prompt
    | model
    | StrOutputParser()
)

In [45]:
summary_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)

summary_chain = (
    summary_prompt
    | model
    | StrOutputParser()
)

In [46]:
language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review?\n\n{Review}"
)

language_chain = (
    language_prompt
    | model
    | StrOutputParser()
)

In [47]:
followup_prompt = ChatPromptTemplate.from_template("""
        Write a follow-up response to the following
        summary in the specified language:
        Summary: {summary}
        Language: {language}
    """)

followup_chain = (
    followup_prompt
    | model
    | StrOutputParser()
)

In [55]:
step_one = RunnableParallel(
    English_Review=translation_chain,
    language=language_chain,
    Review=itemgetter("Review")
)

In [49]:
step_two = (
    step_one
    | RunnablePassthrough.assign(
        summary=lambda x: summary_chain.invoke({
            "English_Review": x["English_Review"]
        })
    )
)

In [50]:
full_chain = (
    step_two
    | RunnablePassthrough.assign(
        followup_message=lambda x: followup_chain.invoke({
            "summary": x["summary"],
            "language": x["language"]
        })
    )
)

In [51]:
review = """
The product quality is excellent and the delivery was very fast.
However, the packaging could be improved.
"""

result = full_chain.invoke({
    "Review": review
})

In [52]:
result

{'English_Review': 'The provided text is already in English. Here is the exact same text:\n\n"The product quality is excellent and the delivery was very fast.\nHowever, the packaging could be improved."',
 'language': 'The review is written in **English**.',
 'Review': '\nThe product quality is excellent and the delivery was very fast.\nHowever, the packaging could be improved.\n',
 'summary': 'The product features excellent quality and fast delivery, though the packaging could be improved.',
 'followup_message': 'Here is a polite and professional follow-up response addressing the review:\n\n"Thank you so much for your wonderful feedback! We are thrilled to hear that you are happy with the excellent quality of the product and our fast delivery. We also truly appreciate your constructive note regarding the packaging—we are actively sharing this with our fulfillment team to ensure we improve your unboxing experience next time. We look forward to serving you again soon!"'}